# Strict-Preference RSD with Nash-Welfare Evaluation


This CPU-ready notebook reuses cached LLM profiles and evaluates strict proxy RSD against oracle RSD using Pareto efficiency, priority blocking, and true Nash welfare.


## 1. Install packages


In [ ]:
!python -m pip install -q -U "transformers>=4.56,<5" "accelerate>=1.2" "bitsandbytes>=0.46.1" scikit-learn scipy sentencepiece

In [ ]:
import ast
import json
import random
import re
import shutil
import time
from itertools import combinations
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import lil_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

SEED = 20260909
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 2. Load the synthetic course-allocation data


In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

import gdown
import numpy as np
import pandas as pd

# Download the dataset ZIP from Google Drive.
FILE_ID = "1Pi_5JOhoCK4pELRhu0iv52us5DCwOJMd"
ZIP_PATH = Path("fixed_tolerance_course_data.zip")

# Download again if the local file is missing or is not a valid ZIP.
if not ZIP_PATH.exists() or not zipfile.is_zipfile(ZIP_PATH):
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()

    downloaded_path = gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )

    assert downloaded_path is not None, (
        "The Google Drive download failed. Make sure the file is shared "
        "with 'Anyone with the link'."
    )

assert ZIP_PATH.exists(), f"Downloaded file not found: {ZIP_PATH}"
assert zipfile.is_zipfile(ZIP_PATH), (
    "The downloaded file is not a valid ZIP. Check the Google Drive "
    "sharing permission and FILE_ID."
)

print(
    f"Dataset ZIP: {ZIP_PATH} "
    f"({ZIP_PATH.stat().st_size / 1_000_000:.2f} MB)"
)

# Extract the dataset into a clean input directory.
INPUT_DIR = Path("fixed_tolerance_input")

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), INPUT_DIR)

# Locate the required files even if the ZIP contains a parent folder.
found = {
    path.name: path
    for path in INPUT_DIR.rglob("*")
    if path.is_file()
}

required = {
    "course_catalog.csv",
    "student_reports.csv",
    "student_ground_truth.json",
    "bundle_queries_with_truth.csv",
    "generation_metadata.json",
}

missing = required - set(found)

assert not missing, (
    f"Missing required files: {sorted(missing)}. "
    f"Files found: {sorted(found)}"
)

# Load the dataset.
catalog = pd.read_csv(found["course_catalog.csv"])
reports_df = pd.read_csv(found["student_reports.csv"])
queries = pd.read_csv(found["bundle_queries_with_truth.csv"])

metadata = json.loads(
    found["generation_metadata.json"].read_text(encoding="utf-8")
)

ground_truth = json.loads(
    found["student_ground_truth.json"].read_text(encoding="utf-8")
)

reports = (
    reports_df
    .set_index("student_id")["language_report"]
    .to_dict()
)

COURSE_IDS = sorted(
    catalog["course_id"].astype(int).tolist()
)

MAX_BUNDLE_SIZE = int(
    metadata["maximum_bundle_size"]
)

TRUE_COMMON_TOLERANCE = float(
    metadata["common_tolerance"]
)

# Convert serialized bundle lists back into Python lists.
for column in ["bundle_a", "bundle_b"]:
    queries[column] = queries[column].apply(json.loads)

# Student splits are generated later inside the mechanism-specific experiment.
student_ids = np.array(sorted(reports_df["student_id"].astype(int).unique()))
print("Students:", len(student_ids))
print("Queries:", len(queries))
print("True tolerance (used only to recover ordinal ground-truth tiers):", TRUE_COMMON_TOLERANCE)

display(catalog)
display(reports_df.sample(3, random_state=SEED))
display(queries.groupby("true_label").size().rename("count"))


In [ ]:
# Restore the previously completed 128-student LLM-profile checkpoint.
# The compressed checkpoint is embedded, so no GPU or LLM call is needed.
import base64, gzip
OUTPUT_DIR = Path("fixed_tolerance_experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "profile_extractions.jsonl"
if not CHECKPOINT.exists():
    CHECKPOINT.write_bytes(gzip.decompress(base64.b64decode("H4sIAAyM8moC/+1dW48dN3J+z68Y6HlOg1Us3vK2wC6CRYwNsLt5CFaG4FgKIERrGbogCSz/9zTZ0tF4m93NJots9tExsIvxjI4t22Sx6qvv8suz9x8+vnz104cXr18+++cH8fgwfuOHDx/fj3/y7O1/Pxv//Od3b//r9ZtX4zd+efbj24/v3r968f7Ht+9evQ/fEf5Tg/8cXL/C8SsIX8nr9+j6lbp+pa9fmetXdvzqMn3Yff1Li+tfET7/bX4dv371vz+/ef3j6w9v/m/6Hb18Mf3+/O/sb/j4IB8f6PFBPT7oxwf7+OAeH0B8P37ux7d///nNq7+P/9ThV/5Nhh/4n7z/+J/vP7z+8PHDq+tP7Pff+7/Xux/+58Xbjx9+/vjB/5v55flPDw/Pv/zr+OGD/8zLV++f+w/5Hz08/HL96euX/tvjb+f5sy+/0H/j+bPv/vivf3j+7NfH+AdA7P2EnX/g93/8y/pn5Pwzf/rDv//1z7/7bvkzlPEZlfEZnfEZt/YZ/5HvH7/8p7uegem/2jP5CcTzZ19+/uQkXH9uxx8//+lXfylevXv39t34/Z8+vnnjz82rH9/+9NKfGYABwRiyRio3/mH0r//0y28vGRRess/XA6/f9LcMI7fs8y/87TWD6zUTsVsm9t2y8ReP/zj/cNfM1+s2/g9iV85/Inrjxr+a4rlxkaP9l7/++d/+9C/f/ceL9Uth9t48kXFSIeO24nG3dfP3lnNbLfcN31FLtz8E+bUEP8m1UgKfVEopUYNz4EihBDH+QWpWSrCwlMQqCVwryef6sf1gb1USTKgkEH2vo8UDwg+aV4+N93rvBzTPZd68mKrniwnbRXr9psEnAIarBoOUwmqLBP7VVjC7ajLrqkGkNY492mxXDdIe7emqxa+XWLxeemqUj32dd/fFsPtqrl6z9fMoNs6jnrrMrQOpB2dQEoCyvvRrOTuPVGFUw9VRDfrrIU2/x1TvPXS7J8TdbWqz1ompH+5mErXN20fDUEN8/6g1OEOoXZhFZ0VEFRYRjDxqF9gN+HytIpBdRbJKyPghFy0hU23hqCCQXUFypj2Z0SD2jOeY45rX/fBc1ZoAn9xaTZhKxmZJoMEJZeX4f873FWbeV+isknCBgsYiVhLguMai25YCGRDh3UNslVIgGgFMplFTcQJQiqOdMAqNJhBmbCastLPaYSq0E7H1ERSNyIndxFQuFpdG/sex1uHgpkG1gKH4j/v626c+6eW3b/PgmsECOT9HG//myTm2Y6vCqGVrT9x9cOUXADV+cGEZ3+HreuvDp7HzlD+a5zSVqZAQbGFCab2bGac5o0CT8qcYIotF13x7jxWPcdY0F9/j1zrSm6dmd8fGNFclXoSCO5RDNmgF8ZgDtxfp49wa2yBlkNMClCMlMTxqEZoBH5lHVBzkLpA8ye0sByqQdzrlGJxpNKtySXXPF5trNEttEVTgFpUuMceaYKUxYtphugi4A9DFhHYBnhEtid1HSxgvI6bT+bxmG6HCredCWsdEU4ENMaBDC1ZgGBD1fE8CyIeKljymgoEYO23/dzJhD34t95NhRX4xlmWtmRpIW+GsFIG1peasLZCN8AYswsl2dGYsGDsf2bo++qA46Svd9FeWZ8T9Vvu4Yu64GpR1bhzmUC6VDj7az2U/yHPZIP5gG+KPXtRruKprupy7kLPWqo/eUMfDpMn4l9iMr9oYJdIbmhSX2OHqgRxqbYzWocOdr+5AVQCPo1TyJ6oUSOUClWm/FkfD5UJyIJG8gVJMMzAHd/dHORPnjhWLLgdQ3KAMjl27leTxEzFfFIKugKlCIutWsD6+05Zw8V643haEusaCcKv6MlCu1DCWXGOd1DqwJmh+qAybtOAC+wWBvDIe8aWh+y0wt6Y1EN2C9Dn9nGhRKr8Jsd2+yyo2WqVEDB0H4/uk8REg3ynZyHW1XezVODjTuQOYXHgfDr6u7iDdbhWM5M7KPoBkLdde/CTBhR3nLCsoLOXJzYuHq0uvXpcRAvciYbV6LCzjulwwIINEsMGU1g0Zu2fAZVkXt7VNLKXkoBLgSIswPsr5+h0F2/o9it7C6ubn0odskxaI05+nzu7W8RVnBeqVrC14pGdVmoIcfLh5I0HreNSELCTAtmAVCuloWiXPhxGEI4kJ7AUFv7DXzUoPsegdxFdAjvUOquIDdAadx4ZtTuKtEQOa8RG2YKR/hzHyDmPVER66p8b6X95hF26rd+GtJn51Z8SWMWJprRSk9OKKjEDSECZxO6f0YQ0S1gXZWVhthc7VC0COSKRhy17lwt0a5evEaucklTPasX0Q1k1/zCsHsVUOWF0CX5ronPes62DJculMxnt439A1F7sZltWc1Ci0lShCXx+ZhhXbJh0S6Rm4OgxjlJ5Rk1i9slE/kF+NDISj+qxIDoQ+H2es0hLYwzH3CrOB4HHq1sP4yFurFE1eSPNyopt7LFaGCRZM8c/psGirV6Fb8zHUt4oWMBkmenc0UmOH4Sbyz7wiGD7v/qpUvcb2aFV5obVW+K1c+3t24Df3MWVbvZXoragG7ZQGsDixf+aWHGiriiz2L+swu3hMZcOv6RJLxLT2v/XpI497snz0pj3ytls4GiG1IRuYJwrnJ89VnYyj9vWXmGH4Z4S8mGSeZxB1lLU989xZ3yr8wJ1XmU4o/c1KX0mciAUPTPkDnsoKDiR5NqsQEJmOpWjlFNy4F06bjU3P7lLYgoUC92Sp9lC6YXGKksPYL6ByMEHpZg6lS6hKkXnSMDS53DtQr4Xm9fA7rVrMt3S/n3VmSErdcaES6OSkQHZuzl2TxVZUqfGR0MigP/nJleEWnp+1Bi1e55MagVcRdakSKhkOYJQ15NDb3js7N+WR8kBYJ+qqiK09eRreygbuGfl604ruybrjxZI4bs18qP+OTvbfGZttpzWiXXjVqYYyLBrF1ZxUVuLc3qsojF8LJu8bnyOIaVv32yWO04osEChaYJtL1YdOC/m79jS5iTlH496TygvurJBSrKxIQyIHZXwAOwYZmbORTGjdKj+zyaMd5us9KZl8jNKGj7MssJlbNwRL4x2qQSvhSKEKhmCRHB9p6toEtO8F1XqQmqu8SOm89Wsfp+bKdwh2EOiU1HZKEoU5WU5avp4HWegubNr07OUgI80Asw+1bNG96BYs8m91zZeKO+BAUliSioLvr4i8Na6CDJYp2CKrg9kZGNejDB4YZPD3BO8+8QYqnEicJSvAmQUvWRK3F8Fr1x28azsVN2we7Y36urAYKTtvhmS9+jOEY4g5iE7QXK91idlzc4Bs+5VaywB6l1KtWkw2fZxC4zbeKGTaeeHgyIE2arqukcwJwkJSesljVcsNUU2d50K7qbp9rDh8y/lfN+Q2CNzqz1T5Q2XHg2/HoYuET+oGPVdjkDxFNG9lC3F7Q9Liu5P4N+Jflk46szzEEZDgiJQIKOuce0bUHL+BnETfqrKuRVVXVcuTxFqiOawxaiFH2Kg2tEogFqetQZtEFeCxPHEKUGkIug8V6clVIwBJZCSEQw6CZJewI7HUjrsegsGBoyEvu5dbLjyKIZpQD5acVgDkdYYC5yQL0qeANJNceJjjTm9lQ17FW57uC472RjyaJ4WLBmcRpZQ2aC4iew7TKq40dW/ZXHLhP+cOSEG+Uae/KtJI1bGL7xn0F2J9cZQefwwknJHjKL0ERdvmwa619qYyUdNcNS2jyPQmjXWPw/gwaNLgZOBRzv+buvY+zdhZUiN2aNiM1fWyTC4TPcVO0nHpSj2uKDn8pJVX7QhfQIJqB+ajpxKn2NMkjZ5yQkJSHS96QE5tO1PhclK/NOM5svJz3sicaKngQBhjx7K7qjl51xvwJkiG4YkUPzVicrPoBwt9wFuvSSHRjZUkgPTzUoLNhxVMHVagRXwZrESIHoZ86IZQfytpaM9dp2hVRYoKgubxJbcKlTAAE5cu0qnKVnZt6zyArVkX2/qS08QMOBkBwN04+6gKbNpqDXOIseuar1wgBiT5uUoJmihsVOSclKiIL9sA9+eaYRfh5Gu0RCa0tFXsQEN96MkSDvLhh2+Em5gcTi4IpLahKXFzAF6pqnNKRlNSxCV6WkqijKLOcHbNYYXHb8XOPrvkPZypLgc0vpXCChlMYPRcdKZ0K7MObGTWcT3kqy+lOptlW77grApT9WZdlxSPpxoMVhsSCDjZ5Mx39spUlXtucIB443QELzGw6m6/ujtq/XiTO//ngMKgeQoDjZOskqSNVQtb2z5StljsmJeLwt2EubTMqHORigscpXM65JQX2hEoJZwOO28dWVS5CjdRsBuAXbDWC40riyquHhlKoq2w15QA2Wgj1Eoc28TarERhhlurqsTH2UpyqOwUYIJzmFm3dyh6YtISJcJACSqURYKx3Q7OPatzVAsvGMOTLphTFNlZAkVOZZalHGi0ciwDGkIk7rxF0NAs+AB7TdQOn1vSr6uutk4VQTR5nHFT/iL/rg7cJ+fZELGrxDAVNA6csBgw+XlNwVbJpE38da7Gw3bH3umcNjrYqzNxfmddzP6yXiE/DtpGUkjinYthtezjvCPneb9dJ6l8jSr/OH/3eerD54kGQQIkKRVA7djcTM2xtCdzM/u2K6tPdksLr1qG+rWgZ9Mrw1Oey1+uymTevIa41cVYErdTW3BoBIbiEekQDogfu2C9LI7PQRwLvBTTrRecOai/jSwEtvInVLndmBlAEaFy0i14O2nN9qhd9neuT8Bg1lGNiltXPjnkscNbFbRVdLy3vYWeNVGmKAeLxgCQokDQitxuwxdVn3q7sRL/d6tl7Uim2NBZv6XAsQ8L/1bYrOAppunrHiahohMKADXaBU2ArmG0JPab8jxhZ0G1orCfuEk15QI5L14tDxwWe9Mq+yHbqCWB4+IQDqV5qpCBlWRVLCxaCQH2dvNK4r5Vc55vx5Zns1dNdecx0igxjqEq0BBmZ8mIRpTh9FcJW7aqi3jJ44MHVPRZnOL0ATTi23rJoAfT/QKnLv+XVaGZTWhWjbKoCZ3yA+xcYGSg+QtzgUrUpKfpxeciEeTA/lmQCvZxjBNnLTM4ENqgBRtcouY0GIOniPbFZPX1Ip1WLqEshwP+UCP8ZWuUl+ujfFrHZK1xUumwjxZuvo82srnKan2hBCy10e3lZ9rT2Y256m2YyTKETuAHpm2dNDpAZwDDsx7p9qmRBCGVcQwMTpx78Ce1lBzAd56PkO67gww8uxEmnU46nIoCqPUcgMT0KxqcBktKkoyH6RlVGEhbkgJwgfQ0PaglWBT189QlAweCn+BrG6n7W3kG09mW14kNAkOQvFclSQAnLQWDXZib+hh9YCF4AgciTx3IkCOJ7pKFmpSBA+XAOSUqXeh9hru9vjdKDfuRgyEEiUaF/j/yypuqK+iobxCmZoftxPrpq1HXwqLZVeNL5xjiOE4YsByg3+Ll6gJarh4sCj+GmvDKuMgrY2uwH0VJQhEWgiip1Kilg8nol8ikQW31ZFQMKDpQg1pwRdP5Bo4hDAgHrVFaOV3WiM+jcYUtYQw0uiB/gMveFfGetlAdyFbOWV3mS+1OGhWj73Lz5ITq5GnRSUTlUAcVA81XIVZUNaI8RG5+fcj3py6cjddsW4hm76KFuqKFRAg4DIdSG9AeApZ2LlqwwCdaqLk2F1nhShkZKlUdzGut0UXHZq10f6O57NnH+6xQWW84FzhcOBccWKwxZK9jucnJSNjAJVosqJCOQXF5nV75Wfl4nAtVTrHpY/Ava9JLQLaxO1fGSg1gfHMu5RzutbK3FJLmgqNubv892eymrn1DR8oSdwwkC4acBo/s6TlR1hKfOwbuzmbuI/hsyTajlvlkFSECa+RZtxa298iznFAaU+iwQ1YaIqW9kYmca0Wsaq8g2/DnEtlzxm6nA7PIGDmQId0yhcIclODcKpqqlQV26n02G2SRZDgQUUphjPX3WkRCpqxuDwfiqrcmNO8NlhQ2PfUGTVztE4vBgXhAW0ZAS3OIqu3BqmYqyZFaayWBKKSzi0gZMVVBCFGShLqzOZD7AlD1gZRRXaMDqKrqVAONTaYzGj2e7SJWBdZ2cZJY0s3kb3ZSiQfqQCUCL8Epn/jYOAoslSrhlXfWL1Wlj+w0FOHe2hosqigYC/1tVxd1pAetYjgleswEK7pznypGbaQMRuAkCTf5uEEkZMOJA+2XdmT7Qa0YL6w5FTW0GMZGK5YDp6LbKwOpwlksG2xo0BalJpxwTzV/0B2wKeUucLDLyG7y4yolevzEwaVAtCA24nGg5O1d631OL5v8ZoA0gjNKZZRCHbxe5uCFw86cFi/NMdBeiZH2xgkV3XQITAFGdZzLDiRg6gEFgbCg/aRAYi6ccrI5YFVTS58d8Nmdg9S9FBwa6Xlj2n0G566xmIBzSkmvtXJOz8ncjpqHh2+A3/nwIfzjkLFAy9LdZoUrVmSaC/52TeM9v7jNFiZluwHIEJIx3rTCREzrnGouMoRaduewe9DGpaQkqulO3TBJhdtyvdH7SbcLnK3KDSnNYloOBhQpcOC7Y4kRGF03744xgxhwgbaz9ZGsAd5dbWecIby1DPtGWuz8WlKequSjUJwE0hDMSVSkOTBdIHRYOQvFv/bmAGO66lnfJS8VrVHbk23TxtELjQClKTBhIYIC2+aG39WIR1fH7434vgVSuz4Rqf3GOUseqFyngqcZ3rtBCGuEJRe8gW2kU3MHuoZfeB3vpzZtDwHcnQmFyKdK5GBx/PSK0/l4p/OsXTlWAjgYCUhCOwii73k23Ph9PkknHO/icn2vFh8rXOqMzpk1Kxrx/1qDhbjerCVOBG5AQnSolX+s0JjIBYBGr5UoyrhIX9l/xQn3vlCHJ1o0cT3ogipXWP3TGjVAFNo6pULslY6cfezCj4/p7O/fNsulweVsr0CTi/MNrJp7dPNbG9qSTDnRWOkoRLlgtArIRlXgAqsMdeTZFV/Nejek+HJpU3Z2iKyRYJ4x7KyTnnPDOTM1VwEGDRg0rrh45ejAxElmTchc3ZUGlJgOH16ofqbFXRByjr02lBnh0GDRSCmnd1eQjBQB1WijnRxq0d4xQ3cYo6b7nl9Pxizd7CNutnXXLBlsehDed0e7EF9tYxBuHjMGmXnjgi2CLZn8Yvg6dcHpiLd/h11EjwCRdJTUQNYnfikIFtAQe5JMoUgxtS+9FGa3YK3GdDHB+pTu7nBs1EtO7b9rIrnMC5KlSTQ4p4Uen5mgJog9MaWUlktRYWCPbuncG7pzTWMONNRzOlwrTeNZQiWSbF6FtuisIl8xgCIlg48FBLtZQFVsTyZIeQvWUtVgrSOYP2cEeauAQaoEDBIDSkvkLErfeZt5MhqAuJ0lTC4YhH2Cwrbj29Nq2XNPZSsEg7DUV9066TyLKeyUIEJkAmgVkhxTSF14jdWvPD6zEozclrtRxeBHH0E6N/kPoS6jI9gBhRpfQoJpNRo5xchmk5UcQiRylOvVVqPqJBYY/A8TdZxFyLSluAELC8VjYeEXpJKsCg+agEgeGYCsYZkHR1nm8Rri3HAdOFm04LllwUc58gGUBQkIrUlZM9numlj1ILZ+GFPpFbWULV/rxgKTkSqTKLDdepQdCN4i/VH5ll6Pj5l22mDQmYzzWeQ4quaOTHX93LPiceImsILNm+ZYq/YqPBzZMfRi7k7yiSDxl/zehJdt7IqFkk6HYBMVw4o1G0vjSV98wcTlSh/FZNFRug1RI2cwrcXyEAxS625oF/eSkm5cnWw9S2DIW+JgSN2KEL/AsI3aPKZ5UG5O/9W1ZHGyPiCKsxb9qkNvPVeDrrY5V3LEXYoB0ZG0jgJVkmJAdd14sWSgWvDEi9GKUs71GFp7iF0P5c+ZjuVkmmE8eM5Q8P11VsVKeR7x5oI8VII+usOlzKEuaLx4oi1/esN7bxAPoRVIpgZRS1QEPtgkUlPwhjLMckvKol5c9hBhZO7Ss7vgtYjjuKF+hyn6OSniWZK1ksJi18SqCRzo0L7hDwj7GB5PeUpbQtcurWZ09TnT1B8bRSMJXpDa6YK9pRsMCqelUhT4TxHePGKjtWWq0AYKt5Zq2YwMF9TfB68sdx9AyaCVy8VetvpDV3BczdgXCimlcuDHTYhs2VHy+UdiRY5ONQ+xqqT1hmt3cSfd3PKYWMg+F1ZIAZoC7ESRnSRS857uAhl14LKrq1tcHtg1y3N3Czk7dZiqBfOJ3bQvT/FEtoOHTqVTJtj3R3QU2Iqoc1nPTuNSLk4v2QFhrdBuKMkBO2sElaThbXrQ4+lzYjxqvpzq2IisD3RmBVa5/VRMfSe1y45bdpBXZg8cjgvtsWXaObQIEiw5zwuIeZSh6SOTdmscTczL2aAEuB6lN44R3KiLg+5YhQLHKnSsnAYQVGhKfRDP/wPdIMkGpcsBAA==")))
print("Cached profile rows:", len(CHECKPOINT.read_text().splitlines()))


## 3. Load the open-source LLM

In [ ]:
CHECKPOINT_READY = Path("fixed_tolerance_experiment_results/profile_extractions.jsonl").exists()
if CHECKPOINT_READY:
    tokenizer = model = None
    MODEL_REVISION = "cached-profile-checkpoint"
    print("Using cached LLM profiles; GPU/model loading skipped.")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    assert torch.cuda.is_available(), "Select a GPU runtime in Colab."
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quantization_config,
        torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
    ).eval()
    MODEL_REVISION = getattr(model.config, "_commit_hash", None)
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Structured profile prompt and robust validation

In [ ]:
PUBLIC_CATALOG = catalog[
    ["course_id", "course_name", "field", "time_slot", "workload", "description"]
].to_dict("records")

PROFILE_PROMPT = '''
You extract a structured course-preference profile from a student's report.
Use only the report and the public course catalog. Never invent preferences.

For every course with clear evidence, assign exactly one attitude from:
STRONGLY_DISLIKE, DISLIKE, NEUTRAL, LIKE, STRONGLY_LIKE.
Omit courses without clear evidence; omitted courses are treated as NEUTRAL.

Extract complements only when two courses are especially valuable together.
Extract substitutes only when the student indicates overlap, redundancy, or a
desire not to take both. Represent a pair as "smaller_id|larger_id".

Return exactly one JSON object:
{
  "course_attitudes": [{"course_id": 3, "attitude": "LIKE"}],
  "complements": ["0|6"],
  "substitutes": ["2|7"]
}
Do not return Markdown, explanations, confidence scores, tolerance estimates,
or additional keys.
'''.strip()

ATTITUDE_VALUES = {
    "STRONGLY_DISLIKE": -2.0,
    "DISLIKE": -1.0,
    "NEUTRAL": 0.0,
    "LIKE": 1.0,
    "STRONGLY_LIKE": 2.0,
}
ATTITUDE_ALIASES = {
    "STRONG_DISLIKE": "STRONGLY_DISLIKE",
    "VERY_DISLIKE": "STRONGLY_DISLIKE",
    "STRONG_LIKE": "STRONGLY_LIKE",
    "VERY_LIKE": "STRONGLY_LIKE",
}


def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("No JSON object found.")
    return json.loads(text[start:end + 1])


def canonical_pair(value):
    if isinstance(value, str):
        pieces = value.split("|")
    elif isinstance(value, (list, tuple)) and len(value) == 2:
        pieces = value
    elif isinstance(value, dict):
        pieces = [value.get("course_id_1"), value.get("course_id_2")]
    else:
        raise ValueError(f"Invalid pair: {value}")
    a, b = sorted(map(int, pieces))
    if a == b or a not in COURSE_IDS or b not in COURSE_IDS:
        raise ValueError(f"Invalid course pair: {value}")
    return (a, b)


def validate_profile(parsed):
    if not isinstance(parsed, dict):
        raise ValueError("The parsed response is not an object.")

    course_scores = {course_id: 0.0 for course_id in COURSE_IDS}
    explicitly_scored = set()
    for item in parsed.get("course_attitudes", []):
        course_id = int(item["course_id"])
        attitude = str(item["attitude"]).strip().upper().replace(" ", "_")
        attitude = ATTITUDE_ALIASES.get(attitude, attitude)
        if course_id not in COURSE_IDS:
            raise ValueError(f"Unknown course ID: {course_id}")
        if attitude not in ATTITUDE_VALUES:
            raise ValueError(f"Unknown attitude: {attitude}")
        course_scores[course_id] = ATTITUDE_VALUES[attitude]
        explicitly_scored.add(course_id)

    complements = sorted(set(canonical_pair(x) for x in parsed.get("complements", [])))
    substitutes = sorted(set(canonical_pair(x) for x in parsed.get("substitutes", [])))
    overlap = set(complements) & set(substitutes)
    if overlap:
        # Ambiguous relationships are conservatively discarded.
        complements = [pair for pair in complements if pair not in overlap]
        substitutes = [pair for pair in substitutes if pair not in overlap]

    return {
        "course_scores": course_scores,
        "explicitly_scored_courses": sorted(explicitly_scored),
        "complements": complements,
        "substitutes": substitutes,
    }


@torch.inference_mode()
def generate_profile(report, max_new_tokens=420):
    payload = {"course_catalog": PUBLIC_CATALOG, "student_report": report}
    messages = [
        {"role": "system", "content": PROFILE_PROMPT},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = output[0, inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    parsed = extract_json_object(raw)
    return validate_profile(parsed), raw

## 5. Smoke test

In [ ]:
if CHECKPOINT_READY:
    print("Smoke test skipped: using the validated profile checkpoint.")
else:
    smoke_student_id = int(reports_df.student_id.iloc[0])
    smoke_profile, smoke_raw = generate_profile(reports[smoke_student_id])
    print(smoke_raw)
    print(smoke_profile)


## 6. Extract one profile per student with checkpointing

In [ ]:
OUTPUT_DIR = Path("fixed_tolerance_experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "profile_extractions.jsonl"

completed = {}
if CHECKPOINT.exists():
    for line in CHECKPOINT.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            completed[int(item["student_id"])] = item
print("Loaded checkpoint rows:", len(completed))

for student_id in tqdm(sorted(map(int, reports)), desc="Extracting profiles"):
    if student_id in completed and completed[student_id].get("status") == "ok":
        continue
    started = time.perf_counter()
    try:
        profile, raw = generate_profile(reports[student_id])
        item = {
            "student_id": student_id,
            "status": "ok",
            "profile": profile,
            "raw_output": raw,
            "error": None,
            "seconds": time.perf_counter() - started,
        }
    except Exception as exc:
        item = {
            "student_id": student_id,
            "status": "error",
            "profile": None,
            "raw_output": None,
            "error": repr(exc),
            "seconds": time.perf_counter() - started,
        }
    completed[student_id] = item
    with CHECKPOINT.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(item) + "\n")

status_df = pd.DataFrame([
    {key: value for key, value in item.items() if key not in {"profile", "raw_output"}}
    for item in completed.values()
]).sort_values("student_id")
display(status_df)
print(status_df.status.value_counts())

failed = status_df.loc[status_df.status != "ok", "student_id"].tolist()
assert not failed, f"Profile extraction failed for students: {failed}. Re-run this cell to retry them."

## 7. Convert profiles and bundle pairs into model features

In [ ]:
profiles = {student_id: item["profile"] for student_id, item in completed.items()}


def bundle_features(profile, bundle):
    bundle = tuple(sorted(map(int, bundle)))
    pairs = set(combinations(bundle, 2))
    score_sum = sum(float(profile["course_scores"].get(str(j), profile["course_scores"].get(j, 0.0))) for j in bundle)
    complement_count = sum(tuple(pair) in pairs for pair in profile["complements"])
    substitute_count = sum(tuple(pair) in pairs for pair in profile["substitutes"])
    explicit_count = sum(j in set(profile["explicitly_scored_courses"]) for j in bundle)
    return np.array([
        score_sum / MAX_BUNDLE_SIZE,
        complement_count / MAX_BUNDLE_SIZE,
        substitute_count / MAX_BUNDLE_SIZE,
        explicit_count / MAX_BUNDLE_SIZE,
    ], dtype=float)


feature_rows = []
for row in queries.itertuples(index=False):
    profile = profiles[int(row.student_id)]
    feature_difference = bundle_features(profile, row.bundle_a) - bundle_features(profile, row.bundle_b)
    feature_rows.append(feature_difference)

X = np.vstack(feature_rows)
FEATURE_NAMES = [
    "course_score_difference",
    "complement_difference",
    "substitute_difference",
    "explicit_coverage_difference",
]
for index, name in enumerate(FEATURE_NAMES):
    queries[name] = X[:, index]

display(queries.head())

The profile-extraction checkpoint uses the same filename as the earlier notebooks. If it is present in the Colab workspace, completed LLM profiles are reused. The allocation experiment below is CPU-only.


## 8. Strict pairwise model, RSD, and true-preference evaluators


In [ ]:
N_REPEATED_SPLITS = 20
N_TRAIN_STUDENTS = 96
N_TEST_STUDENTS = 32
N_RSD_ORDERS = 20
MARKET_CAPACITY_PER_COURSE = 5
LOGIT_C_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

market_capacities = {int(c): MARKET_CAPACITY_PER_COURSE for c in COURSE_IDS}


def strict_pair_target(rows):
    """Binary ordinal target; no utility magnitude and no indifference class."""
    return rows.true_direction.eq("A").astype(int).to_numpy()


def fit_strict_pairwise_model(rows, return_cv=False):
    """Fit a coherent linear bundle score from strict pair directions only."""
    X = rows[FEATURE_NAMES].to_numpy()
    y = strict_pair_target(rows)
    groups = rows.student_id.to_numpy()
    cv = GroupKFold(n_splits=min(N_FOLDS, rows.student_id.nunique()))
    c_rows = []
    for C in LOGIT_C_GRID:
        oof = np.full(len(rows), np.nan)
        for fit_idx, hold_idx in cv.split(X, y, groups):
            model = LogisticRegression(
                C=C, fit_intercept=False, solver="liblinear",
                random_state=SEED, max_iter=2_000,
            )
            model.fit(X[fit_idx], y[fit_idx])
            oof[hold_idx] = model.predict_proba(X[hold_idx])[:, 1]
        predicted = (oof >= 0.5).astype(int)
        c_rows.append({
            "C": C,
            "oof_direction_accuracy": accuracy_score(y, predicted),
            "oof_log_loss": float(-np.mean(
                y * np.log(np.clip(oof, 1e-12, 1))
                + (1 - y) * np.log(np.clip(1 - oof, 1e-12, 1))
            )),
        })
    cv_results = pd.DataFrame(c_rows).sort_values(
        ["oof_direction_accuracy", "oof_log_loss", "C"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    best_C = float(cv_results.loc[0, "C"])
    model = LogisticRegression(
        C=best_C, fit_intercept=False, solver="liblinear",
        random_state=SEED, max_iter=2_000,
    ).fit(X, y)
    return (model, best_C, cv_results) if return_cv else model


def build_candidate_market(rows, model):
    """Construct predicted scores and hidden true values on the same bundle menu."""
    predicted, truth = {}, {}
    for student_id, student_rows in rows.groupby("student_id"):
        i = int(student_id)
        bundles = {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_a
        } | {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_b
        }
        predicted[i] = {
            b: float(np.dot(model.coef_[0], bundle_features(profiles[i], b)))
            for b in bundles
        }
        observations = {b: [] for b in bundles}
        for row in student_rows.itertuples(index=False):
            a = tuple(sorted(map(int, row.bundle_a)))
            b = tuple(sorted(map(int, row.bundle_b)))
            observations[a].append(float(row.true_utility_a))
            observations[b].append(float(row.true_utility_b))
        truth[i] = {b: float(np.mean(values)) for b, values in observations.items()}

        # The outside option is deterministically worst in both rankings.
        predicted[i][tuple()] = min(predicted[i].values()) - 2.0
        truth[i][tuple()] = min(truth[i].values()) - 2.0
    return predicted, truth


def strict_ranks(score_map):
    """Complete strict order; bundle tuple is a report-independent exact-tie rule."""
    ordered = sorted(score_map, key=lambda b: (-score_map[b], b))
    return {bundle: rank for rank, bundle in enumerate(ordered)}


def rank_maps(score_maps):
    return {i: strict_ranks(scores) for i, scores in score_maps.items()}


def fits(bundle, remaining):
    return all(remaining[c] > 0 for c in bundle)


def run_strict_rsd(decision_ranks, capacities, seed):
    rng = np.random.default_rng(seed)
    order = list(map(int, rng.permutation(sorted(decision_ranks))))
    priority_position = {i: position for position, i in enumerate(order)}
    remaining = capacities.copy()
    allocation = {}
    for i in order:
        feasible = [b for b in decision_ranks[i] if fits(b, remaining)]
        chosen = min(feasible, key=lambda b: decision_ranks[i][b])
        allocation[i] = chosen
        for c in chosen:
            remaining[c] -= 1
    return allocation, priority_position


def max_pareto_improvements(allocation, true_ranks, capacities):
    """Max strictly improved students subject to no student becoming worse."""
    students = sorted(true_ranks)
    allowed = {
        i: [b for b in true_ranks[i] if true_ranks[i][b] <= true_ranks[i][allocation[i]]]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []

    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in allowed[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))

    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items():
            matrix[r, col] = value
    objective = np.array([
        -float(true_ranks[i][b] < true_ranks[i][allocation[i]])
        for i, b in variables
    ])
    result = milp(
        c=objective,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        return np.nan, result.message
    improved = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    count = sum(true_ranks[i][improved[i]] < true_ranks[i][allocation[i]] for i in students)
    return int(count), result.message


def priority_blocking_metrics(allocation, true_ranks, capacities, priority_position):
    """Count truly preferred bundles obtainable using empty/lower-priority seats."""
    holders = {c: [] for c in COURSE_IDS}
    for j, bundle in allocation.items():
        for c in bundle:
            holders[c].append(j)

    blocking = []
    for i in sorted(allocation):
        own = allocation[i]
        own_rank = true_ranks[i][own]
        for bundle, rank in true_ranks[i].items():
            if rank >= own_rank:
                continue
            obtainable = True
            for c in bundle:
                if c in own:
                    continue
                course_holders = holders[c]
                if len(course_holders) < capacities[c]:
                    continue
                if any(priority_position[i] < priority_position[j] for j in course_holders):
                    continue
                obtainable = False
                break
            if obtainable:
                blocking.append((i, bundle))
    return {
        "priority_blocking_bundles": len(blocking),
        "priority_blocking_students": len({i for i, _ in blocking}),
        "zero_priority_blocking": int(len(blocking) == 0),
    }


def evaluate_allocation(allocation, true_ranks, capacities, priority_position):
    pe_count, solver_status = max_pareto_improvements(allocation, true_ranks, capacities)
    assigned_ranks = np.array([true_ranks[i][allocation[i]] for i in sorted(allocation)])
    metrics = {
        "pareto_improvable_students": pe_count,
        "pareto_efficient": int(pe_count == 0) if np.isfinite(pe_count) else np.nan,
        "pe_score": 1.0 - pe_count / len(allocation) if np.isfinite(pe_count) else np.nan,
        "mean_true_rank": float(assigned_ranks.mean()),
        "worst_true_rank": int(assigned_ranks.max()),
        "pe_solver_status": solver_status,
    }
    metrics.update(priority_blocking_metrics(
        allocation, true_ranks, capacities, priority_position
    ))
    return metrics


def compare_proxy_to_oracle(proxy_allocation, oracle_allocation, true_ranks):
    students = sorted(true_ranks)
    losses = np.array([
        true_ranks[i][proxy_allocation[i]] - true_ranks[i][oracle_allocation[i]]
        for i in students
    ])
    return {
        "allocation_agreement_rate": float(np.mean([
            proxy_allocation[i] == oracle_allocation[i] for i in students
        ])),
        "mean_true_rank_loss_vs_oracle": float(losses.mean()),
        "median_true_rank_loss_vs_oracle": float(np.median(losses)),
        "share_worse_than_oracle": float(np.mean(losses > 0)),
        "share_better_than_oracle": float(np.mean(losses < 0)),
        "worst_student_rank_loss": int(losses.max()),
    }


def positive_true_values(true_scores, epsilon=1e-4):
    """CEEI-compatible within-student candidate-menu min-max normalization."""
    result = {}
    for i, score_map in true_scores.items():
        nonempty = [float(v) for b, v in score_map.items() if b]
        low, high = min(nonempty), max(nonempty)
        if high - low <= 1e-12:
            values = {b: 1.0 for b in score_map if b}
        else:
            values = {
                b: float(epsilon + (1.0 - epsilon) * (float(v) - low) / (high - low))
                for b, v in score_map.items() if b
            }
        values[tuple()] = float(epsilon)
        result[i] = values
    return result


def nash_welfare_metrics(allocation, true_scores):
    normalized = positive_true_values(true_scores)
    assigned = np.array([normalized[i][allocation[i]] for i in sorted(allocation)], dtype=float)
    log_nw = float(np.mean(np.log(assigned)))
    return {
        "true_log_nash_welfare": log_nw,
        "true_nash_welfare": float(np.exp(log_nw)),
        "mean_normalized_true_utility": float(assigned.mean()),
        "minimum_normalized_true_utility": float(assigned.min()),
        "p10_normalized_true_utility": float(np.quantile(assigned, 0.10)),
    }, normalized


## 9. Repeated strict-RSD experiment


In [ ]:
all_ids = np.array(sorted(queries.student_id.unique()))
assert len(all_ids) == N_TRAIN_STUDENTS + N_TEST_STUDENTS
split_rng = np.random.default_rng(SEED + 1_200)

manifest_rows = []
cv_frames = []
run_rows = []
comparison_rows = []
assigned_rows = []

for split_id in tqdm(range(N_REPEATED_SPLITS), desc="Strict-RSD repeated splits"):
    shuffled = split_rng.permutation(all_ids)
    train_ids = set(map(int, shuffled[:N_TRAIN_STUDENTS]))
    test_ids = set(map(int, shuffled[N_TRAIN_STUDENTS:]))
    assert train_ids.isdisjoint(test_ids)

    for role, ids in [("train", train_ids), ("test", test_ids)]:
        manifest_rows.extend(
            {"split_id": split_id, "student_id": i, "role": role} for i in ids
        )

    train = queries[queries.student_id.isin(train_ids)].copy()
    test = queries[queries.student_id.isin(test_ids)].copy()
    model, best_C, cv_results = fit_strict_pairwise_model(train, return_cv=True)
    cv_results["split_id"] = split_id
    cv_frames.append(cv_results)

    predicted_scores, true_scores = build_candidate_market(test, model)
    predicted_ranks = rank_maps(predicted_scores)
    true_ranks = rank_maps(true_scores)

    for order_run in range(N_RSD_ORDERS):
        order_seed = SEED + 10_000_000 + 100_000 * split_id + order_run
        oracle_allocation, oracle_priority = run_strict_rsd(
            true_ranks, market_capacities, order_seed
        )
        proxy_allocation, proxy_priority = run_strict_rsd(
            predicted_ranks, market_capacities, order_seed
        )
        assert oracle_priority == proxy_priority

        oracle_metrics = evaluate_allocation(
            oracle_allocation, true_ranks, market_capacities, oracle_priority
        )
        proxy_metrics = evaluate_allocation(
            proxy_allocation, true_ranks, market_capacities, proxy_priority
        )
        oracle_nw, normalized_truth = nash_welfare_metrics(oracle_allocation, true_scores)
        proxy_nw, _ = nash_welfare_metrics(proxy_allocation, true_scores)
        oracle_metrics.update(oracle_nw)
        proxy_metrics.update(proxy_nw)
        for method_name, allocation in [
            ("oracle_strict_rsd", oracle_allocation),
            ("llm_proxy_strict_rsd", proxy_allocation),
        ]:
            for i in sorted(allocation):
                assigned_rows.append({
                    "split_id": split_id, "order_run": order_run,
                    "method": method_name, "student_id": i,
                    "assigned_bundle": repr(allocation[i]),
                    "raw_true_utility": true_scores[i][allocation[i]],
                    "normalized_true_utility": normalized_truth[i][allocation[i]],
                })
        run_rows.append({
            "split_id": split_id, "order_run": order_run,
            "method": "oracle_strict_rsd", "best_logit_C": best_C,
            **oracle_metrics,
        })
        run_rows.append({
            "split_id": split_id, "order_run": order_run,
            "method": "llm_proxy_strict_rsd", "best_logit_C": best_C,
            **proxy_metrics,
        })
        comparison_rows.append({
            "split_id": split_id, "order_run": order_run,
            **compare_proxy_to_oracle(proxy_allocation, oracle_allocation, true_ranks),
            "nash_welfare_ratio": float(np.exp(
                proxy_nw["true_log_nash_welfare"] - oracle_nw["true_log_nash_welfare"]
            )),
            "log_nash_welfare_gap": float(
                proxy_nw["true_log_nash_welfare"] - oracle_nw["true_log_nash_welfare"]
            ),
        })

split_manifest = pd.DataFrame(manifest_rows)
cv_results_all = pd.concat(cv_frames, ignore_index=True)
run_results = pd.DataFrame(run_rows)
paired_comparisons = pd.DataFrame(comparison_rows)

summary = (
    run_results.groupby("method", as_index=False)
    .agg(
        pe_rate=("pareto_efficient", "mean"),
        mean_pe_score=("pe_score", "mean"),
        zero_block_rate=("zero_priority_blocking", "mean"),
        mean_blocking_bundles=("priority_blocking_bundles", "mean"),
        median_blocking_bundles=("priority_blocking_bundles", "median"),
        p90_blocking_bundles=("priority_blocking_bundles", lambda x: float(np.quantile(x, 0.90))),
        worst_blocking_bundles=("priority_blocking_bundles", "max"),
        worst_blocking_students=("priority_blocking_students", "max"),
        mean_true_rank=("mean_true_rank", "mean"),
        mean_true_nash_welfare=("true_nash_welfare", "mean"),
        mean_minimum_utility=("minimum_normalized_true_utility", "mean"),
    )
)
comparison_summary = paired_comparisons.describe().T

display(summary)
display(comparison_summary)

oracle_rows = run_results[run_results.method.eq("oracle_strict_rsd")]
oracle_pe_failures = int((oracle_rows.pareto_efficient != 1).sum())
oracle_block_failures = int((oracle_rows.zero_priority_blocking != 1).sum())
print("Oracle PE failure runs:", oracle_pe_failures)
print("Oracle priority-blocking failure runs:", oracle_block_failures)
assert oracle_pe_failures == 0, "Oracle strict RSD failed candidate-domain PE; inspect implementation."
assert oracle_block_failures == 0, "Oracle strict RSD has a priority block; inspect implementation."


## 10. Visualize and save results


In [ ]:
FIGURE_DIR = Path("strict_rsd_figures")
FIGURE_DIR.mkdir(exist_ok=True)

proxy_runs = run_results[run_results.method.eq("llm_proxy_strict_rsd")]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(proxy_runs.priority_blocking_bundles, bins=20, color="#2563eb", alpha=0.80)
axes[0].set(
    title="True priority-blocking bundles under LLM-proxy RSD",
    xlabel="Blocking bundles", ylabel="Number of RSD runs",
)
values = np.sort(proxy_runs.priority_blocking_bundles.to_numpy())
axes[1].step(values, np.arange(1, len(values) + 1) / len(values), where="post", color="#7c3aed")
axes[1].set(
    title="Empirical CDF of blocking bundles",
    xlabel="Blocking bundles", ylabel="Cumulative probability",
)
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "proxy_blocking_distribution.png", dpi=180)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(proxy_runs.pareto_improvable_students, bins=range(0, 34), color="#dc2626", alpha=0.75)
axes[0].set(
    title="Candidate-domain Pareto violations",
    xlabel="Pareto-improvable students", ylabel="Number of RSD runs",
)
axes[1].hist(paired_comparisons.allocation_agreement_rate, bins=20, color="#059669", alpha=0.80)
axes[1].set(
    title="Proxy agreement with oracle strict RSD",
    xlabel="Student-level bundle agreement", ylabel="Number of paired runs",
)
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "proxy_pe_and_agreement.png", dpi=180)
plt.show()

split_robustness = (
    proxy_runs.groupby("split_id", as_index=False)
    .agg(
        mean_blocks=("priority_blocking_bundles", "mean"),
        worst_blocks=("priority_blocking_bundles", "max"),
        pe_rate=("pareto_efficient", "mean"),
    )
)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(split_robustness))
ax.bar(x - 0.2, split_robustness.mean_blocks, width=0.4, label="Mean blocks")
ax.bar(x + 0.2, split_robustness.worst_blocks, width=0.4, label="Worst blocks")
ax.set(
    xlabel="Repeated train-test split", ylabel="Blocking bundles",
    title="Robustness across student samples",
)
ax.set_xticks(x); ax.set_xticklabels(split_robustness.split_id)
ax.grid(axis="y", alpha=0.25); ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "split_level_robustness.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.hist(paired_comparisons.nash_welfare_ratio, bins=20, color="#0f766e", alpha=0.82)
ax.axvline(1.0, color="black", linestyle="--")
ax.set(title="Strict RSD: proxy/oracle true Nash welfare", xlabel="Nash-welfare ratio", ylabel="Paired runs")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "strict_rsd_nash_welfare_ratio.png", dpi=180)
plt.show()

OUTPUT_DIR = Path("strict_rsd_llm_robustness_results")
OUTPUT_DIR.mkdir(exist_ok=True)
split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
cv_results_all.to_csv(OUTPUT_DIR / "pairwise_model_cv.csv", index=False)
run_results.to_csv(OUTPUT_DIR / "rsd_runs.csv", index=False)
pd.DataFrame(assigned_rows).to_csv(OUTPUT_DIR / "assigned_student_utilities.csv", index=False)
paired_comparisons.to_csv(OUTPUT_DIR / "proxy_oracle_paired_comparisons.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
comparison_summary.to_csv(OUTPUT_DIR / "comparison_summary.csv")
split_robustness.to_csv(OUTPUT_DIR / "split_robustness.csv", index=False)
for path in FIGURE_DIR.glob("*.png"):
    shutil.copy2(path, OUTPUT_DIR / path.name)

metadata = {
    "seed": SEED,
    "mechanism": "strict-preference random serial dictatorship over candidate bundles",
    "training_information": "pairwise direction A/B only",
    "excluded_from_training": [
        "utility magnitude", "indifference labels", "tier threshold",
        "allocation utility", "Pareto violations", "blocking counts",
    ],
    "domain": "observed candidate bundles plus outside option",
    "nash_welfare": "geometric mean of within-student min-max normalized true assigned utilities",
    "utility_normalization": "candidate-menu min-max to [1e-4,1], outside fixed at 1e-4",
    "scope_warning": "PE and blocking claims are relative to this candidate-bundle domain.",
    "n_repeated_splits": N_REPEATED_SPLITS,
    "split_sizes": {"train": N_TRAIN_STUDENTS, "test": N_TEST_STUDENTS},
    "rsd_orders_per_test_market": N_RSD_ORDERS,
    "capacity_per_course": MARKET_CAPACITY_PER_COURSE,
    "oracle_pe_failures": oracle_pe_failures,
    "oracle_priority_blocking_failures": oracle_block_failures,
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR))
print("Saved:", archive.resolve())
from google.colab import files
files.download(str(archive))
